# Data exploration - Strand Bookstore
This notebook contains an exploration of the data shape for used books in the Strand Bookstore's inventory.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
url ="https://www.strandbooks.com/books-and-media.html"

In [90]:
driver = webdriver.Chrome()
driver.get(url)

wait = WebDriverWait(driver, timeout=30)

old_text = wait.until(lambda driver: driver.find_element(By.CLASS_NAME, "product-item")).text

condition_used_good_btn = wait.until(EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Used - Good')]")))

driver.execute_script("arguments[0].scrollIntoView();", condition_used_good_btn)     

condition_used_good_btn.click()

wait.until(lambda driver: driver.find_element(By.CLASS_NAME, "product-item").text != old_text)

html_source = driver.page_source
driver.quit()

In [88]:
soup=BeautifulSoup(html_source,'lxml')

Each product card has the class name "product-info".

In [105]:
product_list = soup.find_all("div", class_="product-info")

In [108]:
data = []
for product in product_list:
    text = product.get_text(separator="\n", strip=True)

    stock_status = "Out of Stock" if "Out of Stock" in text else "In Stock"
    link = product.find("a").get('href')

    title = product.find("a", class_="product-item-link").get_text(strip=True)

    if not title:
        continue

    author = product.find(
        "div",
        class_="mb-1.5 items-center justify-center font-normal text-sm text-center"
    ).get_text(strip=True)

    price = product.find("span", class_="price").get_text(strip=True)

    product_data = [title, link, author, price, stock_status]
    data.append(product_data)
print(f"Items processed: {len(data)}")


Items processed: 24


In [111]:
df = pd.DataFrame(data)
df.head(15)

,0,1,2,3,4
0,Halloween: Costumes and Other Treats,https://www.strandbooks.com/halloween-costumes...,"Schneider, Stuart",$18.00,Out of Stock
1,Getting to Reparations: How Building a Differe...,https://www.strandbooks.com/getting-to-reparat...,"Brown, Dorothy A.",$18.00,In Stock
2,"The Snakes That Ate Florida: Reporting, Essays...",https://www.strandbooks.com/the-snakes-that-at...,"Frazier, Ian",$19.00,In Stock
3,Deserving: What the Lives of the Condemned Rev...,https://www.strandbooks.com/deserving-what-the...,"Vartkessian, Elizabeth|Prejean, Sister Helen",$15.00,In Stock
4,There Is No Place for Us: Working and Homeless...,https://www.strandbooks.com/there-is-no-place-...,"Goldstone, Brian",$12.00,In Stock
5,The House of Beauty: Lessons from the Image In...,https://www.strandbooks.com/the-house-of-beaut...,"Sicardi, Arabelle",$18.00,In Stock
6,The First Shots: The Epic Rivalries and Heroic...,https://www.strandbooks.com/first-shots-the-ep...,"Borrell, Brendan",$10.00,In Stock
7,The Last Supper: How to Overcome the Coming Fo...,https://www.strandbooks.com/last-supper-how-to...,"Kass, Sam",$24.00,In Stock
8,Anointed: The Extraordinary Effects of Social ...,https://www.strandbooks.com/anointed-the-extra...,"Stuart, Toby",$17.50,In Stock
9,Mexico's Day of the Dead: A Celebration of Lif...,https://www.strandbooks.com/mexico-s-day-of-th...,"Navarro, Luisa|Chitnis, Christine",$28.00,In Stock
